In [1]:
!pip -q install lime reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.3 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import glob
import textwrap
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from lime.lime_text import LimeTextExplainer

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

In [3]:
if not os.path.exists("election-dataset-us"):
    !git clone https://github.com/ai4society/election-dataset-us.git
else:
    print("Repo already exists.")

Cloning into 'election-dataset-us'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 84 (delta 9), reused 63 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 594.93 KiB | 8.15 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [4]:
DATA_DIR = "election-dataset-us/data-us"

def clean_text(x):
    if x is None:
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def detect_label_from_source(source_value):
    s = clean_text(source_value).lower()
    if "vote411" in s:
        return 1
    return 0

def extract_state_from_filename(path):
    base = os.path.basename(path)
    # example: ak_qa.json -> ak
    return base.split("_")[0].upper()

def get_possible(record, keys, default=""):
    for k in keys:
        if k in record and record[k] is not None:
            return record[k]
    return default

def normalize_record(record, state_code):
    # likely field names
    q = get_possible(record, ["q", "question", "Q", "Question"])
    a = get_possible(record, ["a", "answer", "A", "Answer"])
    s = get_possible(record, ["s", "source", "url", "Source"])

    q = clean_text(q)
    a = clean_text(a)
    s = clean_text(s)

    text = clean_text(f"{q} {a}")

    if text == "":
        return None

    label = detect_label_from_source(s)

    return {
        "State": state_code,
        "text": text,
        "label": label
    }

rows = []

json_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
print("JSON files found:", len(json_files))

for jf in json_files:
    state_code = extract_state_from_filename(jf)

    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    # handle list or dict-wrapped content
    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        # try common container names
        if "data" in data and isinstance(data["data"], list):
            records = data["data"]
        elif "qas" in data and isinstance(data["qas"], list):
            records = data["qas"]
        elif "faq" in data and isinstance(data["faq"], list):
            records = data["faq"]
        else:
            # fallback: if dict itself looks like one record
            records = [data]
    else:
        records = []

    for rec in records:
        if isinstance(rec, dict):
            row = normalize_record(rec, state_code)
            if row is not None:
                rows.append(row)

df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

print("Total rows:", len(df))
print(df.head())
print(df["label"].value_counts(dropna=False))

JSON files found: 50
Total rows: 103
  State                                               text  label
0    NM  ['How do I register to vote in New Mexico?', '...      0
1    TX  I’m not sure if I’m registered; how can I conf...      0
2    TX  I’m not registered, but want to vote in the No...      0
3    TX  If I send my registration by the deadline, wha...      0
4    TX  I am registered to vote, but I moved this past...      0
label
0    63
1    40
Name: count, dtype: int64


In [5]:
combined_csv = "us-allstates-voter-faqs.csv"
df.to_csv(combined_csv, index=False)
print(f"Saved: {combined_csv}")

Saved: us-allstates-voter-faqs.csv


In [6]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 82
Test size: 21


In [7]:
models = {
    "logreg": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )),
        ("clf", LogisticRegression(max_iter=2000, random_state=42))
    ]),

    "svm_calibrated": Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95
        )),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(random_state=42),
            cv=5
        ))
    ])
}

In [8]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results.append({
        "model": name,
        "f1": f1,
        "auc_roc": auc
    })

    print(f"\nModel: {name}")
    print("F1:", round(f1, 4))
    print("AUC ROC:", round(auc, 4))
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values(["auc_roc", "f1"], ascending=False).reset_index(drop=True)
results_df


Model: logreg
F1: 0.4
AUC ROC: 0.9808
              precision    recall  f1-score   support

           0       0.68      1.00      0.81        13
           1       1.00      0.25      0.40         8

    accuracy                           0.71        21
   macro avg       0.84      0.62      0.61        21
weighted avg       0.80      0.71      0.66        21


Model: svm_calibrated
F1: 0.875
AUC ROC: 0.9904
              precision    recall  f1-score   support

           0       0.92      0.92      0.92        13
           1       0.88      0.88      0.88         8

    accuracy                           0.90        21
   macro avg       0.90      0.90      0.90        21
weighted avg       0.90      0.90      0.90        21



,model,f1,auc_roc
0,svm_calibrated,0.875,0.990385
1,logreg,0.400,0.980769


In [9]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print("Best model:", best_model_name)

Best model: svm_calibrated


In [10]:
sample_rows = []
missing_notes = []

for state in sorted(df["State"].unique()):
    state_df = df[df["State"] == state]

    primary_rows = state_df[state_df["label"] == 0]
    secondary_rows = state_df[state_df["label"] == 1]

    if len(primary_rows) > 0:
        sample_rows.append(primary_rows.sample(1, random_state=42))
    else:
        missing_notes.append(f"{state}: missing primary row")

    if len(secondary_rows) > 0:
        sample_rows.append(secondary_rows.sample(1, random_state=42))
    else:
        missing_notes.append(f"{state}: missing secondary row")

sample_df = pd.concat(sample_rows).reset_index(drop=True)

print("Selected rows:", len(sample_df))
print("Missing notes:")
for note in missing_notes:
    print("-", note)

Selected rows: 3
Missing notes:
- NM: missing secondary row


In [11]:
class_names = ["primary", "secondary"]
explainer = LimeTextExplainer(class_names=class_names)

def get_top3_lime_features(text, model):
    exp = explainer.explain_instance(
        text_instance=text,
        classifier_fn=model.predict_proba,
        num_features=10,
        labels=[0, 1]
    )

    pred_class = int(model.predict([text])[0])
    top_feats = exp.as_list(label=pred_class)[:3]

    # pad if fewer than 3
    while len(top_feats) < 3:
        top_feats.append(("", 0.0))

    return exp, top_feats

output_rows = []
lime_objects = []

for _, row in sample_df.iterrows():
    text = row["text"]
    state = row["State"]
    true_class = int(row["label"])

    pred_class = int(best_model.predict([text])[0])
    pred_prob = float(best_model.predict_proba([text])[0][pred_class])

    exp, top_feats = get_top3_lime_features(text, best_model)
    lime_objects.append((state, text, true_class, pred_class, pred_prob, exp))

    output_rows.append({
        "Text (Q/A)": text,
        "state": state,
        "predicted class": pred_class,
        "true class": true_class,
        "feature_1": top_feats[0][0],
        "weight_1": top_feats[0][1],
        "feature_2": top_feats[1][0],
        "weight_2": top_feats[1][1],
        "feature_3": top_feats[2][0],
        "weight_3": top_feats[2][1]
    })

sample_output_df = pd.DataFrame(output_rows)
sample_output_df.head()

,Text (Q/A),state,predicted class,true class,feature_1,weight_1,feature_2,weight_2,feature_3,weight_3
0,"['How do I register to vote in New Mexico?', '...",NM,0,0,ballot,-0.238263,party,0.095155,clerk,0.078911
1,How do I Apply to be a Student Election Clerk?...,TX,0,0,clerk,0.049654,Clerk,0.034583,student,0.025657
2,Voters can call or text 844-338-8743 at any ti...,TX,1,1,days,0.055319,counted,0.050597,Voters,0.042356


In [12]:
sample_csv_name = f"us-allstates-voter-faqs-classifier-{best_model_name}-sampleoutput.csv"
sample_output_df.to_csv(sample_csv_name, index=False)
print(f"Saved: {sample_csv_name}")

Saved: us-allstates-voter-faqs-classifier-svm_calibrated-sampleoutput.csv


In [13]:
pdf_name = f"us-allstates-voter-faqs-classifier-{best_model_name}-sampleoutput-dump.pdf"

styles = getSampleStyleSheet()
doc = SimpleDocTemplate(pdf_name, pagesize=letter)
story = []

story.append(Paragraph(f"LIME Explanations Dump - {best_model_name}", styles["Title"]))
story.append(Spacer(1, 12))

if missing_notes:
    story.append(Paragraph("Missing data notes:", styles["Heading2"]))
    for note in missing_notes:
        story.append(Paragraph(note, styles["BodyText"]))
    story.append(Spacer(1, 12))

for i, (state, text, true_class, pred_class, pred_prob, exp) in enumerate(lime_objects, start=1):
    story.append(Paragraph(f"Example {i}", styles["Heading2"]))
    story.append(Paragraph(f"State: {state}", styles["BodyText"]))
    story.append(Paragraph(f"True class: {true_class} ({class_names[true_class]})", styles["BodyText"]))
    story.append(Paragraph(f"Predicted class: {pred_class} ({class_names[pred_class]})", styles["BodyText"]))
    story.append(Paragraph(f"Predicted probability for predicted class: {pred_prob:.4f}", styles["BodyText"]))
    story.append(Spacer(1, 6))

    wrapped_text = "<br/>".join(textwrap.wrap(text[:2500], width=110))
    story.append(Paragraph(f"Text: {wrapped_text}", styles["BodyText"]))
    story.append(Spacer(1, 6))

    feats = exp.as_list(label=pred_class)
    story.append(Paragraph("LIME explanation:", styles["Heading3"]))
    for feat, wt in feats[:10]:
        story.append(Paragraph(f"{feat}: {wt:.4f}", styles["BodyText"]))

    story.append(Spacer(1, 12))
    if i % 2 == 0:
        story.append(PageBreak())

doc.build(story)
print(f"Saved: {pdf_name}")

Saved: us-allstates-voter-faqs-classifier-svm_calibrated-sampleoutput-dump.pdf


In [14]:
from google.colab import files

files.download("us-allstates-voter-faqs.csv")
files.download(sample_csv_name)
files.download(pdf_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>